# PIA-Net v2 — Standalone Notebook (two real bugs fixed)

Age-r version-e PIA-Net **chance-level** perform korchilo. Ekhon karon-ta khuje ber kora hoyeche —
duita real bug chilo, duitai ei notebook-e fix kora:

---

### Bug 1 — Model matro **428 sample** diye train hocchilo (asol karon)

| | Training data actually seen |
|---|---|
| UNet / ResNet (reference number gulo) | `training_data_generator` = **infinite stream**, 10,000 fresh sample **proti epoch** x 500 epoch = **~5,000,000 sample presentation** (paper Table I) |
| PIA-Net (age-r notebook) | `val_data.npz`-er P=16 slice = **428 sample**, 24 bar repeat |

Prai **4 order of magnitude** kom data. Tomar age-r training log-e ei problem-tai spashto dekha jay:
train loss 0.0071 kintu val loss 0.0126 — 428-ta sample mukhosto kore felechilo, generalize kichui korte
pareni. Ei notebook ekhon **paper-er ঐ same infinite generator** use kore (Section IV-B), tai protiti
batch-i fresh, kokhono repeat hoy na.

### Bug 2 — Physics dictionary-r steering vector-e **sign ulta** chilo

Repo-r asol `ev()` function use kore `exp(-j*pi*cos(angle)*k)`, kintu age-r notebook-e inline kora
version-e `exp(+j*pi*n*cos(angle))` lekha chilo — mane **complex conjugate**, ar codebook-o
simplified `linspace` diye banano chilo asol DFT-based (`arccos`) construction-er bodole.
Fole physics/ISTA branch-ta data-r shathe match-i korchilo na — oi branch kar্যত garbage feed korchilo.
Ekhon repo-r **exact** function gulo inline kora ache, tai dictionary ekdom data generation-er shathe mile.

---

### Ei run-e ki dekhbe (honest expectation)

Age-r result je **chance level** chilo tar proof: RMSE shob SNR-e prai flat **~0.579**, ar uniform
[0,1] degree error-er RMS = 1/sqrt(3) = **0.5774** — hubohu meleche. Mane "detected" angle gulo
random hit chilo, real estimate na.

Ei notebook-e Part 14-e ekta **automatic "real learning na chance?" test** ache. Eta dekhbe:
1. RMSE 0.577-er theke spashto niche namche kina, ar
2. SNR barle Pd barche kina (monotonic trend)।

Duita-i pass korle bujhbe model shotti shikhche. Na hole notebook nijei bole dibe — kono
overclaim korbe na.

## Part 0 — Setup + imports

In [ ]:
import importlib, subprocess, sys

def ensure(pip_name, import_name=None):
    import_name = import_name or pip_name
    try:
        importlib.import_module(import_name)
        print(f'{import_name}: already available, skip.')
    except ImportError:
        print(f'{import_name}: not found, installing {pip_name} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pip_name], check=True)

for pip_name, import_name in [('opencv-python-headless', 'cv2')]:
    ensure(pip_name, import_name)
print('Dependency check done.')

In [ ]:
import os, math, random, pickle, time

import numpy as np
import scipy.ndimage as ndi
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt
import cv2

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
from tensorflow.keras import layers as L

np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## Part 1 — Exact physics functions (repo-r `dldoa_dataset_generation.py` theke hubohu)

**Eta-i Bug 2-er fix.** Ei cell-er protita function repo-r original file theke **exact copy** —
steering vector-er sign, DFT codebook-er `arccos` construction, channel model, noise scaling,
ground-truth Gaussian — shob। Age simplified/vul version chilo, tar fole physics branch data-r
shathe match korto na.

In [ ]:
def wrapTo2Pi(x):
    return np.mod(x, 2 * np.pi)

def ev(nt, angle):
    """Steering vector, paper Eqs. (2)-(3):
    a(phi) = (1/sqrt(nt)) * [1, e^{-j*pi*cos(phi)}, ..., e^{-j*pi*(nt-1)*cos(phi)}]^T
    NOTE the MINUS sign in the exponent -- age-r notebook-e eta plus chilo (bug)."""
    k = np.arange(nt)
    vector = (1 / np.sqrt(nt)) * np.exp(-1j * np.pi * np.cos(angle) * k)
    return vector[:, np.newaxis]

def beamforming_vector_generation_P(P, nt):
    """TX DFT codebook F (nt x P), Eq.(16) of TSDCE."""
    p = np.arange(P)
    cosp = (1 / np.pi) * np.angle(np.exp(1j * (2 * np.pi / P) * p))
    phi_p = np.arccos(cosp)
    F = np.zeros((nt, P), dtype=complex)
    for idx_p in range(P):
        F[:, idx_p] = np.squeeze(ev(nt, phi_p[idx_p]), -1)
    return F

def beamforming_vector_generation_Q(Q, nr):
    """RX DFT combining matrix W (nr x Q)."""
    q = np.arange(Q)
    cosq = (1 / np.pi) * np.angle(np.exp(-1j * (2 * np.pi / Q) * q))
    phi_q = np.arccos(cosq)
    W = np.zeros((nr, Q), dtype=complex)
    for idx_q in range(Q):
        W[:, idx_q] = np.squeeze(ev(nr, phi_q[idx_q]), -1)
    return W

def generate_points(M_pts, delta, max_attempts=10000, rng=None):
    """M random (AoD, AoA) points in [0,pi]^2 with min pairwise separation delta."""
    if rng is None:
        rng = np.random.default_rng()
    points = []
    attempts = 0
    while len(points) < M_pts and attempts < max_attempts:
        x = rng.uniform(0, math.pi); y = rng.uniform(0, math.pi)
        if not any(math.hypot(x - p[0], y - p[1]) < delta for p in points):
            points.append((x, y))
        attempts += 1
    if len(points) < M_pts:
        raise ValueError(f'Could not place {M_pts} points with delta={delta}')
    return points

def generate_noise(var_alpha, SNR, Q, P, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    var_noise = var_alpha * 10 ** (-SNR / 10)
    sigma_n = np.sqrt(var_noise / 2)
    return sigma_n * (rng.standard_normal((Q, P)) + 1j * rng.standard_normal((Q, P)))

def generate_channel_v2(nr, nt, angle_v, alpha_l):
    """H = sqrt(nt*nr) * sum_l alpha_l * a_r(psi_l) * a_t^H(phi_l)   (Eq. 1)"""
    Lp = len(alpha_l)
    Hl = np.zeros((nr, nt, Lp), dtype=complex)
    for l in range(Lp):
        qq = ev(nt, angle_v[l]).conj().T      # (1, nt)
        ww = ev(nr, angle_v[l + Lp])          # (nr, 1)
        Hl[:, :, l] = alpha_l[l] * (ww * qq)
    return np.sum(np.sqrt(nt * nr) * Hl, axis=-1)

def gaus2d(dist_m, dist_n, sigma):
    coeff = 1.0 / (2.0 * np.pi * sigma ** 2)
    return coeff * np.exp(-(dist_m ** 2 + dist_n ** 2) / (2.0 * sigma ** 2))

def generate_gt(Lp, amps, f1, f2, num_points_rx=256, num_points_tx=256,
                sigma=0.07, margin_factor=3.0):
    """Ground-truth heatmap = sum of L 2-D Gaussians (Eq. 11)."""
    f1 = wrapTo2Pi(np.asarray(f1)); f2 = wrapTo2Pi(np.asarray(f2))
    margin = margin_factor * sigma
    p = np.linspace(-margin, 2 * np.pi + margin, num_points_tx, endpoint=False)
    q = np.linspace(-margin, 2 * np.pi + margin, num_points_rx, endpoint=False)
    Wp, Wq = np.meshgrid(p, q)
    J = np.zeros_like(Wp)
    for l in range(Lp):
        J = J + amps[l] * gaus2d(Wp - f1[l], Wq - f2[l], sigma)
    return J

def get_real_imag(H):
    return np.dstack((np.real(H), np.imag(H)))

print('Exact physics functions loaded.')

## Part 2 — Constants + codebooks + the physics dictionary

PIA-Net ekhane **P=Q=16** condition-er jonno specialized (paper-er Figs. 5-6 test condition-o thik
eta-i)। Ei choice-er ekta boro subidha: raw observation shorashori 16x16, tai 64x64-e zoom kore abar
16x16-e recover korar lossy round-trip pura baad — training-e kono interpolation artifact nei।

In [ ]:
NT = NR = P_CB = Q_CB = 16
G_GRID = 32          # angle-grid resolution for the physics dictionary
M_OUT = 256          # heatmap resolution (paper)
SIGMA_GT = 0.07      # paper

F_CB = beamforming_vector_generation_P(P_CB, NT)   # (nt, P)
W_CB = beamforming_vector_generation_Q(Q_CB, NR)   # (nr, Q)

# angle grid for the sparse dictionary
psis = np.linspace(0.05, np.pi - 0.05, G_GRID)
phis = np.linspace(0.05, np.pi - 0.05, G_GRID)
A_r = np.hstack([ev(NR, a) for a in psis])         # (nr, G)
A_t = np.hstack([ev(NT, a) for a in phis])         # (nt, G)

# Observation model:  G = W^H H F = c * sum_l alpha_l (W^H a_r)(a_t^H F)
#   -> U = W^H A_r  (Q x G),   V = F^T conj(A_t)  (P x G),   G ~ c * U X V^T
U_dict = (W_CB.conj().T @ A_r).astype(np.complex64)
V_dict = (F_CB.T @ A_t.conj()).astype(np.complex64)
DICT_C = np.float32(np.sqrt(NT * NR))

print('U_dict:', U_dict.shape, ' V_dict:', V_dict.shape, ' c =', DICT_C)

fig, axs = plt.subplots(1, 2, figsize=(9, 3.2))
axs[0].imshow(np.abs(U_dict), aspect='auto', cmap='viridis'); axs[0].set_title('|U_dict| (Rx)')
axs[1].imshow(np.abs(V_dict), aspect='auto', cmap='viridis'); axs[1].set_title('|V_dict| (Tx)')
plt.tight_layout(); plt.show()

### Part 2.1 — Dictionary sanity check (eta age fail korto)

Ekta noiseless single-path observation banai, tarpor dictionary diye matched-filter kore dekhi
peak-ta **shothik angle-e** poড়ে kina. Sign vul thakle peak vul jaygay porto — tai eta Bug 2-er
direct verification.

In [ ]:
# one noiseless path at a known angle
test_phi, test_psi = 1.1, 2.0
H_t = generate_channel_v2(NR, NT, np.array([test_phi, test_psi]), np.array([1.0 + 0j]))
G_t = W_CB.conj().T @ H_t @ F_CB

# matched filter against the dictionary:  score = |U^H G conj(V)|
score = np.abs(U_dict.conj().T @ G_t @ V_dict.conj())
i_hat, j_hat = np.unravel_index(np.argmax(score), score.shape)

print(f'true  psi={test_psi:.3f} rad, phi={test_phi:.3f} rad')
print(f'found psi={psis[i_hat]:.3f} rad, phi={phis[j_hat]:.3f} rad')
print(f'grid spacing = {psis[1]-psis[0]:.3f} rad -> error should be < 1 grid cell')
ok = abs(psis[i_hat]-test_psi) < 2*(psis[1]-psis[0]) and abs(phis[j_hat]-test_phi) < 2*(phis[1]-phis[0])
print('DICTIONARY CHECK:', 'PASS -- physics matches the data model' if ok else 'FAIL -- dictionary mismatched!')

plt.figure(figsize=(4,3.5))
plt.imshow(score, cmap='hot', origin='lower')
plt.scatter([j_hat],[i_hat], marker='x', c='cyan', s=80)
plt.title('Matched-filter score (should peak at true angle)')
plt.xlabel('phi grid'); plt.ylabel('psi grid'); plt.tight_layout(); plt.show()

## Part 3 — Infinite training generator (Bug 1-er fix)

**Eta-i shobcheye important change.** Paper Section IV-B-er exact spec:
- L (path count): random {1..9}
- SNR: random {-15..24} dB
- alpha_l ~ CN(0, 1/L), strongest first
- AoA/AoD uniform [0,pi], minimum separation pi/6
- P=Q=16, nt=nr=16 (ei notebook P=16-e specialized)

Protiti sample **fresh** — kokhono repeat hoy na, tai 428-sample overfitting-er problem-i thake na.
Ground truth per-sample max=1 e normalize kora (karon evaluation-er blob detector shob shomoy
`cv2.normalize` kore, tai absolute amplitude irrelevant — model-ke shudhu location shikhte dile
kaj onek shoja hoy)।

In [ ]:
def pia_sample_generator(sigma=SIGMA_GT, M=M_OUT):
    """Infinite generator: yields (raw 16x16x2 observation, normalized 256x256x1 heatmap)."""
    rng = np.random.default_rng()
    while True:
        Lp = rng.integers(1, 10)             # 1..9 paths
        SNR = rng.integers(-15, 25)          # -15..24 dB

        alpha_l = np.sqrt(1.0 / Lp) * (rng.standard_normal(Lp) + 1j * rng.standard_normal(Lp)) / np.sqrt(2)
        alpha_l = alpha_l[np.argsort(np.abs(alpha_l))[::-1]]      # strongest first

        pts = generate_points(int(Lp), np.pi / 6, rng=rng)
        phi_l = [p[0] for p in pts]          # AoD
        psi_l = [p[1] for p in pts]          # AoA
        angle_v = np.hstack([phi_l, psi_l])

        omega_phi = np.pi * np.cos(phi_l)
        omega_psi = -np.pi * np.cos(psi_l)

        H = generate_channel_v2(NR, NT, angle_v, alpha_l)
        Gm = W_CB.conj().T @ H @ F_CB
        Z = generate_noise(1.0, float(SNR), Q_CB, P_CB, rng=rng)
        Y = Gm + Z                            # (Q, P) = (16, 16)

        data = get_real_imag(Y).astype(np.float32)             # (16,16,2)

        gt = generate_gt(int(Lp), np.ones(Lp), omega_phi, omega_psi,
                         num_points_rx=M, num_points_tx=M, sigma=sigma)
        mx = gt.max()
        if mx > 0:
            gt = gt / mx                       # per-sample max-normalize
        gt = np.real(gt)[..., None].astype(np.float32)          # (256,256,1)

        yield data, gt

# quick look at one generated sample
_g = pia_sample_generator()
_d, _gt = next(_g)
print('generated data:', _d.shape, ' gt:', _gt.shape, ' gt max:', _gt.max())
fig, axs = plt.subplots(1, 3, figsize=(11, 3.2))
axs[0].imshow(_d[:, :, 0], cmap='viridis'); axs[0].set_title('raw observation (real)')
axs[1].imshow(_d[:, :, 1], cmap='viridis'); axs[1].set_title('raw observation (imag)')
axs[2].imshow(_gt[:, :, 0], cmap='hot');    axs[2].set_title('ground-truth heatmap (normalized)')
for ax in axs: ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

### Part 3.1 — `tf.data` pipeline

Generation-i bottleneck (~40 sample/sec ekta CPU thread-e), tai kayekta parallel worker + prefetch
use kora hocche jate GPU boshe na thake.

In [ ]:
BATCH_SIZE = 32
STEPS_PER_EPOCH = 125        # 125 x 32 = 4000 fresh samples per epoch
NUM_WORKERS = 4              # parallel generator workers

output_sig = (
    tf.TensorSpec(shape=(P_CB, Q_CB, 2), dtype=tf.float32),
    tf.TensorSpec(shape=(M_OUT, M_OUT, 1), dtype=tf.float32),
)

def _make_ds(_):
    return tf.data.Dataset.from_generator(pia_sample_generator, output_signature=output_sig)

train_ds = (
    tf.data.Dataset.range(NUM_WORKERS)
    .interleave(_make_ds, cycle_length=NUM_WORKERS,
                num_parallel_calls=tf.data.AUTOTUNE, deterministic=False)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

# fixed validation batch (generated once, so val_loss is comparable epoch to epoch)
_vg = pia_sample_generator()
_vx, _vy = zip(*[next(_vg) for _ in range(256)])
X_gen_val = np.stack(_vx); Y_gen_val = np.stack(_vy)
print('validation-from-generator:', X_gen_val.shape, Y_gen_val.shape)

t0 = time.time()
for _b in train_ds.take(3):
    pass
print(f'pipeline warm-up ok, ~{(time.time()-t0)/3:.2f}s per batch of {BATCH_SIZE}')

## Part 4 — PIA-Net architecture

Architecture age-r motoi (specially icche kore un-changed rakha hoyeche) — jate bujha jay
**shudhu data + dictionary fix korle** same architecture kaj kore kina। Eta-i valo experimental
hygiene: ek shathe shob change korle bujha jeto na kon-ta kaj korlo.

- **Path A (physics)**: Learned-ISTA — deep-unfolded sparse recovery, protiti iteration-e trainable
  step-size o threshold, ar prottek iteration-e value clip (numerical divergence protection)।
- **Path B (learned)**: raw observation-er upor shorashori CNN, 32x32-e upsample — physics branch
  converge korte na parleo model attke jay na।
- Duita fuse -> self-attention refine -> convolutional decoder -> 256x256, shathe ekta
  "physics skip" (ISTA map shorashori final layer porjonto)।

In [ ]:
# ---- operator normalization (Bug 3 fix) ----
# ISTA converges only if step < 1/L, where L is the Lipschitz constant of the
# gradient = c^2 * sigma_max(U)^2 * sigma_max(V)^2. For this dictionary L ~ 12089,
# so the required step is < 8.3e-5 -- but the previous code used 0.06, i.e. 725x too
# large. That diverges by ~724x PER ITERATION (~1e17x over 6 iterations); the value
# clip was only hiding it, leaving an 85%-zero / 13%-saturated binary map with zero
# gradient on both sides. Rescaling U and V to unit spectral norm and normalizing Y
# per sample makes the whole problem scale-free (L = 1), so step ~0.9 is correct and
# the trainable parameters live at O(1) -- which also matters because Adam at
# lr=1.5e-3 could never meaningfully tune a parameter whose correct value is 7e-5.
_sU = np.linalg.svd(U_dict, compute_uv=False)[0]
_sV = np.linalg.svd(V_dict, compute_uv=False)[0]
U_norm = (U_dict / _sU).astype(np.complex64)
V_norm = (V_dict / _sV).astype(np.complex64)
print(f'sigma_max(U)={_sU:.4f}  sigma_max(V)={_sV:.4f}')
print(f'old Lipschitz L = {(DICT_C**2)*(_sU**2)*(_sV**2):.0f}  -> required step < {1/((DICT_C**2)*(_sU**2)*(_sV**2)):.2e}')
print('after normalization: L = 1, so step ~0.9 is correct and scale-free.')

MAX_ISTA_VAL = 10.0   # pure safety net now; the healthy range is ~0-0.6

def _inv_softplus(v):
    return float(np.log(np.expm1(v)))

class LearnedISTA(tf.keras.layers.Layer):
    def __init__(self, U_dict, V_dict, n_iters=6, **kwargs):
        super().__init__(**kwargs)
        self.U = tf.constant(U_dict, dtype=tf.complex64)
        self.V = tf.constant(V_dict, dtype=tf.complex64)
        self.n_iters = n_iters
        self.G = U_dict.shape[1]

    def build(self, input_shape):
        # softplus reparametrization keeps the trainable variables at O(1) scale
        # (softplus(0.3783)=0.9, softplus(-4.6)=0.01) so Adam can actually tune them,
        # and guarantees step/threshold stay positive.
        self.s = [self.add_weight(name=f'step_{k}', shape=(), dtype=tf.float32,
                                   initializer=tf.keras.initializers.Constant(_inv_softplus(0.9)))
                  for k in range(self.n_iters)]
        self.t = [self.add_weight(name=f'thresh_{k}', shape=(), dtype=tf.float32,
                                   initializer=tf.keras.initializers.Constant(_inv_softplus(0.01)))
                  for k in range(self.n_iters)]

    def call(self, y_real_imag):
        Y = tf.complex(y_real_imag[..., 0], y_real_imag[..., 1])
        batch = tf.shape(Y)[0]
        # per-sample normalization -> scale-free and robust across SNR / path count
        nrm = tf.norm(tf.reshape(tf.abs(Y), (batch, -1)), axis=1)
        nrm = tf.reshape(nrm, (-1, 1, 1)) + 1e-9
        Yn = Y / tf.complex(nrm, tf.zeros_like(nrm))

        X = tf.zeros((batch, self.G, self.G), dtype=tf.float32)
        Uc = tf.math.conj(self.U); Vc = tf.math.conj(self.V)
        for k in range(self.n_iters):
            Xc = tf.cast(X, tf.complex64)
            Yhat = tf.einsum('qi,bij,pj->bqp', self.U, Xc, self.V)
            R = Yn - Yhat
            grad = tf.math.real(tf.einsum('qi,bqp,pj->bij', Uc, R, Vc))
            step = tf.nn.softplus(self.s[k])
            th = tf.nn.softplus(self.t[k])
            X = tf.nn.relu(X + step * grad - th)
            X = tf.clip_by_value(X, 0.0, MAX_ISTA_VAL)
        return tf.expand_dims(X, axis=-1)

def attention_refine_block(x, d_model=48, n_heads=4):
    shape = x.shape[1:3]
    h = L.Conv2D(d_model, 1, padding='same')(x)
    seq = L.Reshape((shape[0] * shape[1], d_model))(h)
    attn = L.MultiHeadAttention(num_heads=n_heads, key_dim=d_model // n_heads)(seq, seq)
    seq = L.LayerNormalization()(L.Add()([seq, attn]))
    h = L.Reshape((shape[0], shape[1], d_model))(seq)
    h = L.Conv2D(d_model, 1, padding='same', activation='relu')(h)
    return L.Add()([x, h])

def conv_block(x, filters):
    return L.Conv2D(filters, 3, padding='same', activation='relu')(x)

def build_pia_net(g_grid=G_GRID, m_out=M_OUT, n_ista_iters=6):
    inputs = tf.keras.Input(shape=(P_CB, Q_CB, 2), name='raw_observation')

    # normalized dictionary + no downstream rescale: the ISTA map is already O(0-0.6)
    ista_map = LearnedISTA(U_norm, V_norm, n_iters=n_ista_iters, name='learned_ista')(inputs)

    b = L.Conv2D(32, 3, padding='same', activation='relu')(inputs)
    b = L.Conv2D(32, 3, padding='same', activation='relu')(b)
    b = L.Conv2DTranspose(32, 3, strides=2, padding='same', activation='relu')(b)
    b = L.Conv2D(32, 3, padding='same', activation='relu')(b)

    x = L.Concatenate()([ista_map, b])
    x = conv_block(x, 48)
    x = attention_refine_block(x, d_model=48, n_heads=4)
    x = conv_block(x, 64)
    x = conv_block(x, 64)

    for _ in range(int(np.log2(m_out // g_grid))):
        x = L.Conv2DTranspose(64, 3, strides=2, padding='same', activation='relu')(x)
        x = conv_block(x, 64)

    physics_skip = L.Resizing(m_out, m_out, interpolation='bilinear')(ista_map)
    x = L.Concatenate()([x, physics_skip])
    x = conv_block(x, 32)
    outputs = L.Conv2D(1, 3, padding='same', activation='sigmoid', name='heatmap',
                        kernel_initializer=tf.keras.initializers.RandomNormal(stddev=5e-3),
                        bias_initializer=tf.keras.initializers.Constant(-3.0))(x)
    return tf.keras.Model(inputs, outputs, name='PIA-Net-v2')

pia_net = build_pia_net()
PIA_PARAM_COUNT = pia_net.count_params()
print('PIA-Net total params:', f'{PIA_PARAM_COUNT:,}')
pia_net.summary()

### Part 4.1 — ISTA health check (Bug 3 verification)

Ei cell dekhbe physics branch **shotti kaj korche kina**. Age (step=0.06) ISTA diverge korto:
output-er 85% exactly 0, 13.5% clip-e saturated — mane practically ekta binary mask, duidike-i
gradient zero, tai oi branch kichui shikhte parto na.

Ekhon ja dekha uchit:
- `frac exactly 0` prai **90-99%** (sparse hওয়া-i thik, kintu 100% na)
- `max` prai **0.05-1.0** (clip 10.0-e kokhono na thaka)
- noiseless single-path test-e peak **shothik angle-e**

In [ ]:
_probe = LearnedISTA(U_norm, V_norm, n_iters=6)
_xs = np.stack([next(pia_sample_generator())[0] for _ in range(32)])
_o = _probe(tf.constant(_xs)).numpy()
print(f'ISTA output on 32 real samples:')
print(f'  max={_o.max():.4f}   mean={_o.mean():.5f}   frac exactly 0 = {(_o<=0).mean()*100:.1f}%')
print(f'  frac AT clip ({MAX_ISTA_VAL}) = {(_o>=MAX_ISTA_VAL*0.999).mean()*100:.3f}%   (should be 0.000%)')

# noiseless single-path localization
_tphi, _tpsi = 1.1, 2.0
_H = generate_channel_v2(NR, NT, np.array([_tphi, _tpsi]), np.array([1.0+0j]))
_Y = W_CB.conj().T @ _H @ F_CB
_xin = get_real_imag(_Y).astype(np.float32)[None, ...]
_X = _probe(tf.constant(_xin)).numpy()[0, :, :, 0]
_i, _j = np.unravel_index(np.argmax(_X), _X.shape)
_err = np.hypot(psis[_i]-_tpsi, phis[_j]-_tphi)
print(f'\nnoiseless 1-path: peak psi={psis[_i]:.3f} (true {_tpsi:.3f}), phi={phis[_j]:.3f} (true {_tphi:.3f})')
print(f'  error = {_err:.3f} rad  (grid spacing {psis[1]-psis[0]:.3f})')
print('  ISTA HEALTH:', 'PASS -- physics branch is informative' if _err < 0.2 and _o.max() < MAX_ISTA_VAL*0.99 else 'FAIL')

fig, axs = plt.subplots(1, 2, figsize=(9, 3.5))
axs[0].imshow(_X, cmap='hot', origin='lower'); axs[0].scatter([_j],[_i], marker='x', c='cyan', s=90)
axs[0].set_title('ISTA sparse code (noiseless 1 path)'); axs[0].set_xlabel('phi'); axs[0].set_ylabel('psi')
axs[1].imshow(_o[0,:,:,0], cmap='hot', origin='lower'); axs[1].set_title('ISTA on a real noisy sample')
plt.tight_layout(); plt.show()

## Part 5 — Loss + compile

In [ ]:
def weighted_mse(alpha=20.0):
    def loss_fn(y_true, y_pred):
        w = 1.0 + alpha * y_true
        return tf.reduce_mean(w * tf.square(y_pred - y_true))
    return loss_fn

pia_net.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1.5e-3, clipnorm=5.0),
    loss=weighted_mse(alpha=20.0),
)
print('Compiled.')

## Part 6 — Training on the infinite stream

`EPOCHS x STEPS_PER_EPOCH x BATCH_SIZE` = total **fresh, never-repeated** samples.
Default: 60 x 125 x 32 = **240,000 unique samples** (age chilo 428 — prai **560x** beshi)।

Time lagbe mota-muti: generation ~40 sample/sec/worker, 4 worker -> ~1 min/epoch, tai ~60 min।
Tomar hate shomoy thakle `EPOCHS` ba `STEPS_PER_EPOCH` bariye dao — ei model-er jonno data
beshi dile-i shobcheye beshi labh (paper-er UNet 5,000,000 sample dekhecho)।

In [ ]:
EPOCHS = 60

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=12,
                                      restore_best_weights=True, min_delta=1e-5),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                          patience=5, min_lr=1e-5, verbose=1),
    tf.keras.callbacks.TerminateOnNaN(),
]

t0 = time.time()
history = pia_net.fit(
    train_ds,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_data=(X_gen_val, Y_gen_val),
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=2,
)
n_ep = len(history.history['loss'])
print(f'Training shesh: {n_ep} epoch, {time.time()-t0:.0f}s')
print(f'Total FRESH samples seen: {n_ep * STEPS_PER_EPOCH * BATCH_SIZE:,}')

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.xlabel('Epoch'); plt.ylabel('Weighted MSE'); plt.title('PIA-Net v2 training curve')
plt.legend(); plt.grid(alpha=0.3); plt.show()

print('NOTE: ekhon train ar val loss kache kache thaka uchit -- infinite fresh data,')
print('tai age-r motto 428-sample overfitting (train 0.0071 vs val 0.0126) ar hওয়ar kotha na.')

## Part 7 — Tomar real test set load koro (evaluation-er jonno)

Training ekhon generator theke hocche, kintu **evaluation tomar Kaggle dataset-er asol test set
diye-i** hobe — jate hardcoded UNet/ResNet reference number-er shathe apple-to-apple compare hoy।

In [ ]:
def find_dataset_dir(search_roots, markers=('test_data.npz', 'val_data.npz', 'train_data.npz')):
    for root in search_roots:
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            if any(m in filenames for m in markers):
                return dirpath
    return None

DATA_DIR = find_dataset_dir(['/kaggle/input', '/content', '.'])
if DATA_DIR is None:
    DATA_DIR = '/kaggle/input/dl-doa/content/DL_DOA_CLONE/dataset'   # fallback
print('DATA_DIR =', DATA_DIR)

def _p(*names):
    for n in names:
        q = os.path.join(DATA_DIR, n)
        if os.path.exists(q):
            return q
    return None

def _load_features(path):
    if path is None:
        return None
    if path.endswith('.pkl'):
        with open(path, 'rb') as f:
            return pickle.load(f)
    return np.load(path, allow_pickle=True)

tp, gp, mp = _p('test_data.npz'), _p('test_gt.npz'), _p('test_meta.npz')
if tp and gp and mp:
    X_test = np.load(tp)['data']; Y_test = np.load(gp)['data']; meta_test = np.load(mp)['data']
    feat_test = _load_features(_p('test_features.npy', 'test_features.pkl'))
    print('X_test:', X_test.shape, ' meta_test:', meta_test.shape)
else:
    X_test = Y_test = meta_test = feat_test = None
    print('test set paoa jayni -- evaluation skip hobe.')

### Part 7.1 — 64x64 stored input theke raw 16x16 recover koro

Dataset-e input 64x64-e zoom kore rakha (nearest-neighbour, order=0)। Model raw 16x16 nay, tai
exact index-mapping diye reverse kora hocche — order=0 hওয়ay eta **lossless**।

In [ ]:
def build_recovery_index(raw_size=16, zoom_factor=4):
    idx_map = ndi.zoom(np.arange(raw_size), zoom_factor, order=0)
    return np.array([np.where(idx_map == i)[0][0] for i in range(raw_size)])

_recovery_idx = build_recovery_index(16, 4)

def recover_raw_batch(data_64):
    return data_64[:, _recovery_idx, :, :][:, :, _recovery_idx, :]

if X_test is not None:
    X_test_raw = recover_raw_batch(X_test)
    print('X_test_raw:', X_test_raw.shape)
    # lossless check: re-zoom and compare
    _chk = ndi.zoom(X_test_raw[0, :, :, 0], 4, order=0)
    print('recovery lossless:', np.allclose(_chk, X_test[0, :, :, 0]))
else:
    X_test_raw = None

## Part 8 — Inference utilities (blob detection -> angles -> RMSE / Pd)

In [ ]:
def prepare_prediction_for_peaks(prediction):
    m = prediction[:, :, 0].numpy() if hasattr(prediction, 'numpy') else prediction[:, :, 0]
    return cv2.normalize(m, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

def get_blob_detector():
    p = cv2.SimpleBlobDetector_Params()
    p.filterByColor = True; p.blobColor = 255
    p.minThreshold = 0; p.maxThreshold = 255
    p.filterByArea = True; p.minArea = 1; p.maxArea = 1000
    p.filterByCircularity = False; p.filterByConvexity = False; p.filterByInertia = False
    return cv2.SimpleBlobDetector_create(p)

detector = get_blob_detector()

def reorder_keypoints(keypoints, img_norm):
    coords = np.array([kp.pt for kp in keypoints])
    if len(coords) == 0:
        return [], np.array([])
    cr = np.round(coords).astype(int)
    amps = []
    for (x, y) in cr:
        amps.append(img_norm[y, x] if (0 <= y < img_norm.shape[0] and 0 <= x < img_norm.shape[1]) else 0)
    amps = np.array(amps); order = np.argsort(-amps)
    return [keypoints[i] for i in order], amps[order]

def get_blob_peaks(pred, detector):
    img = prepare_prediction_for_peaks(pred)
    kps = detector.detect(img)
    kps, amps = reorder_keypoints(kps, img)
    peaks = np.array([kp.pt for kp in kps]) if len(kps) else np.zeros((0, 2))
    return peaks, amps

def wrap_2pi_to_minus_pi(a):
    a = np.asarray(a)
    return np.where(a > np.pi, a - 2 * np.pi, a)

def peaks_to_angles(peaks, margin_factor=3.0, sigma=0.07, grid_size=256):
    if peaks.shape[0] == 0:
        return np.array([]), np.array([])
    margin = margin_factor * sigma
    pxy = peaks.T
    ext = 2 * np.pi + 2 * margin
    fe = -margin + (pxy / grid_size) * ext
    fm = wrap_2pi_to_minus_pi(fe)
    return np.arccos(-fm[1] / np.pi), np.arccos(fm[0] / np.pi)

def permute_pairs(A, B):
    A = np.asarray(A); B = np.asarray(B)
    d = np.linalg.norm(A[:, None, :] - B[None, :, :], axis=2)
    r, c = linear_sum_assignment(d)
    return [(tuple(A[i]), tuple(B[j])) for i, j in zip(r, c)]

def prepare_for_metric(angles_est, feat):
    Lp = feat.shape[-1]
    if len(angles_est[0]) < Lp:
        return np.array([feat[0], feat[1]]), np.array([np.full((Lp,), np.nan), np.full((Lp,), np.nan)])
    ae = (angles_est[0][:Lp], angles_est[1][:Lp])
    perm = permute_pairs(list(zip(feat[0], feat[1])), list(zip(ae[0], ae[1])))
    psi_t, phi_t = zip(*[p[0] for p in perm])
    psi_e, phi_e = zip(*[p[1] for p in perm])
    return np.array([psi_t, phi_t]), np.array([psi_e, phi_e])

def get_ang_difference(gt_angles, pred_angles):
    H = np.angle(np.exp(1j * gt_angles) * np.exp(-1j * pred_angles))
    return (H * (180 / np.pi)).flatten()

def filter_angles(d, max_deg_error=1.0):
    return d[np.abs(d) <= max_deg_error], d[np.abs(d) > max_deg_error]

print('Inference utilities ready.')

## Part 9 — Single example, stage by stage

In [ ]:
def show_single(model, X_raw, Y, feat, idx):
    pred = tf.squeeze(model(tf.expand_dims(X_raw[idx], 0), training=False), axis=0)
    peaks, amps = get_blob_peaks(pred, detector)
    Lp = feat.shape[-1]
    peaks_s = peaks[np.argsort(-amps)[:Lp]] if len(peaks) else peaks

    gt = Y[idx, :, :, 0]
    gt = gt / max(gt.max(), 1e-9)
    fig, axs = plt.subplots(1, 3, figsize=(13, 4))
    axs[0].imshow(X_raw[idx][:, :, 0], cmap='viridis'); axs[0].set_title('raw observation (16x16)')
    axs[1].imshow(gt, cmap='hot'); axs[1].set_title('ground truth')
    axs[2].imshow(pred.numpy()[:, :, 0], cmap='hot'); axs[2].set_title('PIA-Net prediction')
    if len(peaks_s):
        axs[2].scatter(peaks_s[:, 0], peaks_s[:, 1], c='cyan', marker='x', s=70)
    gy, gx = np.unravel_index(np.argmax(gt), gt.shape)
    axs[2].scatter([gx], [gy], facecolors='none', edgecolors='lime', s=200, linewidths=2)
    for a in axs: a.set_xticks([]); a.set_yticks([])
    plt.tight_layout(); plt.show()
    print('cyan x = predicted peaks, green circle = TRUE strongest peak')

if X_test_raw is not None:
    show_single(pia_net, X_test_raw, Y_test, feat_test[0], 0)
    show_single(pia_net, X_test_raw, Y_test, feat_test[14], 14)

## Part 10 — Full evaluation on your test set

In [ ]:
def evaluate_pia_net(model, X_raw, meta, feat, batch=64):
    preds = model.predict(X_raw, batch_size=batch, verbose=0)
    results = {}
    for i in range(len(X_raw)):
        Lp, SNR, QP = int(meta[i, 0]), int(meta[i, 1]), int(meta[i, 2])
        peaks, amps = get_blob_peaks(preds[i], detector)
        peaks = peaks[np.argsort(-amps)[:Lp]] if len(peaks) else peaks
        ang = peaks_to_angles(peaks, sigma=0.07, grid_size=preds.shape[1])
        gt_a, pr_a = prepare_for_metric(ang, feat[i])
        results.setdefault((Lp, SNR, QP), []).append((gt_a, pr_a))

    rmse, pd_ = {}, {}
    for cond, ex in results.items():
        good, bad = [], []
        for gt_a, pr_a in ex:
            if np.isnan(pr_a).any():
                bad.append(np.full(gt_a.size, 999.0)); continue
            g, b = filter_angles(get_ang_difference(gt_a, pr_a), 1.0)
            good.append(g); bad.append(b)
        good = np.concatenate(good) if good else np.array([])
        bad = np.concatenate(bad) if bad else np.array([])
        tot = len(good) + len(bad)
        rmse[cond] = np.sqrt(np.mean(good ** 2)) if len(good) else np.nan
        pd_[cond] = len(good) / tot if tot else np.nan
    return rmse, pd_

if X_test_raw is not None:
    t0 = time.time()
    pia_rmse, pia_pd = evaluate_pia_net(pia_net, X_test_raw, meta_test, feat_test)
    print(f'evaluated {len(X_test_raw)} samples in {time.time()-t0:.1f}s')
    for c in sorted(pia_rmse, key=lambda k: k[1]):
        print(c, f'RMSE={pia_rmse[c]:.4f}', f'Pd={pia_pd[c]:.4f}')
else:
    pia_rmse, pia_pd = {}, {}

## Part 11 — Hardcoded baseline reference (UNet / ResNet)

Ei number gulo ei repo-r **committed, actually-run reproduction** theke
(`DL_DOA/figures_unet/*.pkl`, `DL_DOA/figures_resnet/*.pkl`) — protibar heavy baseline
retrain korte hobe na.

In [ ]:
UNET_REF_RMSE = {(3,-10,16):0.5498,(3,-5,16):0.5069,(3,0,16):0.4548,(3,5,16):0.3777,
                 (3,10,16):0.3047,(3,15,16):0.2556,(3,20,16):0.2297,(3,25,16):0.2086}
UNET_REF_PD   = {(3,-10,16):0.2226,(3,-5,16):0.4706,(3,0,16):0.6788,(3,5,16):0.8127,
                 (3,10,16):0.8883,(3,15,16):0.9244,(3,20,16):0.9439,(3,25,16):0.9509}
RESNET_REF_RMSE = {(3,-10,16):0.5532,(3,-5,16):0.5118,(3,0,16):0.4581,(3,5,16):0.3920,
                   (3,10,16):0.3256,(3,15,16):0.2792,(3,20,16):0.2528,(3,25,16):0.2377}
RESNET_REF_PD   = {(3,-10,16):0.2043,(3,-5,16):0.4366,(3,0,16):0.6396,(3,5,16):0.7790,
                   (3,10,16):0.8623,(3,15,16):0.8987,(3,20,16):0.9250,(3,25,16):0.9378}
UNET_PARAMS, RESNET_PARAMS = 31_276_481, 469_393
print('Reference table loaded.')

## Part 12 — THE HONEST TEST: real learning, na chance?

Eta ei notebook-er shobcheye important cell. Duita objective test:

**Test A — RMSE floor test.** Jodi "detected" angle gulo random hit hoy, tader error [0,1] degree-e
uniform hoy, ar tar RMS = 1/sqrt(3) = **0.5774**. Real estimator-er error 0-er kache jome, tai RMSE
spashto **0.5774-er niche** thake (UNet: 0.21-0.55)।

**Test B — SNR monotonicity.** Signal quality barle Pd barte **hobei**. Flat thakle bujhte hobe
model signal-i porche na।

In [ ]:
CHANCE_RMSE = 1.0 / np.sqrt(3)   # 0.5774

if pia_rmse:
    snrs_p = sorted(s for (l, s, q) in pia_rmse if l == 3 and q == 16)
    r_vals = np.array([pia_rmse[(3, s, 16)] for s in snrs_p], dtype=float)
    p_vals = np.array([pia_pd[(3, s, 16)] for s in snrs_p], dtype=float)

    mean_rmse = np.nanmean(r_vals)
    # Test A
    testA = mean_rmse < 0.52                      # clearly below the chance floor
    # Test B: Pd should increase with SNR
    valid = ~np.isnan(p_vals)
    corr = np.corrcoef(np.array(snrs_p)[valid], p_vals[valid])[0, 1] if valid.sum() > 2 else np.nan
    pd_rise = np.nanmax(p_vals) - np.nanmin(p_vals)
    testB = (corr > 0.7) and (pd_rise > 0.15)

    print('='*70)
    print('TEST A -- RMSE floor')
    print(f'  chance floor (uniform errors) = {CHANCE_RMSE:.4f}')
    print(f'  your mean RMSE               = {mean_rmse:.4f}')
    print(f'  -> {"PASS: below chance floor, real estimation" if testA else "FAIL: at chance level (random hits)"}')
    print()
    print('TEST B -- Pd rises with SNR')
    print(f'  corr(SNR, Pd) = {corr:.3f}   Pd range = {pd_rise:.3f}')
    print(f'  -> {"PASS: model responds to signal quality" if testB else "FAIL: flat in SNR, model ignores the signal"}')
    print('='*70)
    if testA and testB:
        print('VERDICT: PIA-Net is genuinely learning. Now the accuracy gap vs UNet is a')
        print('         real, meaningful comparison -- report it as-is.')
    elif testB and not testA:
        print('VERDICT: model responds to SNR (partially learning) but precision still at')
        print('         the chance floor -- needs more training data/epochs.')
    else:
        print('VERDICT: STILL AT CHANCE LEVEL. Do NOT report this as a working method.')
        print('         Next lever: raise EPOCHS/STEPS_PER_EPOCH (more fresh samples) --')
        print('         data volume is the single biggest factor for this task.')
    print('='*70)
else:
    print('no evaluation available.')

## Part 12.5 — Gap-ta KOTHAY? (recall na precision)

Pd proti sample-e 6-ta angle gone (3 path x psi,phi)। Kintu metric-er ekta guruttopurno
byapar ache: blob detector jodi L-er kom blob paay, `prepare_for_metric` shob NaN ferot dey
ar **puro sample 0/6 pay** -- 4/6 na. Tai Pd=0.62 duita ekdom alada jinish bojhate pare:

- **(a)** 3-ta blob paoa jacche kintu ekta 1 degree-r baire -> **PRECISION** shomossha
- **(b)** proyoi 3-ta blob-i paoa jacche na -> puro sample zero -> **RECALL** shomossha

Duitar shomadhan ekdom alada, tai **age mapo, tarpor bodlao**.

In [ ]:
n_blobs_hist = {}
n_short = 0
err_when_enough = []
good_when_enough = 0
total_when_enough = 0

preds_d = pia_net.predict(X_test_raw, batch_size=64, verbose=0)

for i in range(len(X_test_raw)):
    Lp = int(meta_test[i, 0])
    peaks, amps = get_blob_peaks(preds_d[i], detector)
    nb = len(peaks)
    n_blobs_hist[nb] = n_blobs_hist.get(nb, 0) + 1
    if nb < Lp:
        n_short += 1
        continue
    peaks_s = peaks[np.argsort(-amps)[:Lp]]
    ang = peaks_to_angles(peaks_s, sigma=0.07, grid_size=preds_d.shape[1])
    gt_a, pr_a = prepare_for_metric(ang, feat_test[i])
    d = get_ang_difference(gt_a, pr_a)
    err_when_enough.append(np.abs(d))
    g, b = filter_angles(d, 1.0)
    good_when_enough += len(g)
    total_when_enough += len(g) + len(b)

N = len(X_test_raw)
print('='*72)
print('BLOB COUNT DISTRIBUTION (detector koyta peak pelo)')
for k in sorted(n_blobs_hist):
    print(f'   {k:3d} blobs : {n_blobs_hist[k]:5d} samples ({n_blobs_hist[k]/N*100:5.1f}%)')
print()
print(f'L-er kom blob paoa samples : {n_short}/{N} = {n_short/N*100:.1f}%')
print('   ^ egulo automatically 0/6 pay, heatmap joto valo-i hok')
print()
if total_when_enough:
    pd_enough = good_when_enough / total_when_enough
    errs = np.concatenate(err_when_enough)
    print('SHUDHU jekhane >= L blob paoa geche:')
    print(f'   Pd (precision-only)      = {pd_enough:.4f}')
    print(f'   median |angle error|     = {np.median(errs):.4f} deg')
    print(f'   fraction of angles <1deg = {(errs<1.0).mean():.4f}')
    print(f'   fraction of angles <0.5  = {(errs<0.5).mean():.4f}')
    print()
    print('INTERPRETATION:')
    if n_short / N > 0.20:
        print(f'   -> RECALL-i main loss: {n_short/N*100:.0f}% sample-e L-ta blob-i paoa jay na.')
        print('      Peak extraction / blob sharpness age thik koro -- shobcheye shosta labh.')
    elif pd_enough < 0.75:
        print('   -> PRECISION main loss: blob paoa jay kintu 1 degree-r baire pore.')
        print('      Finer angular resolution (boro G_GRID) / better decoder lagbe.')
    else:
        print('   -> Duita-i thik ache; baki gap mulotoi angular precision.')
print('='*72)

## Part 12.6 — DARK sub-pixel decoding (**retrain lagbe na**)

Mapa bottleneck: PIA-Net-er high-SNR RMSE 0.431 deg = ei 256-grid-er **0.90 pixel** --
mane integer-pixel quantization limit-e atke ache. UNet paay 0.209 deg = **0.44 pixel**
(sub-pixel), karon tar blob gulo joth eshto mosrin je detector-er centroid sub-pixel-e pore.

[DARK (Zhang et al., CVPR 2020)](https://arxiv.org/abs/1910.06278) thik eta-i thik kore,
**shudhu decoding bodle** -- model-e hat dite hoy na. Gaussian blob log-space-e quadratic
hoye jay, tai peak-er charpashe 2nd-order Taylor expansion sub-pixel offset-ta bodhdho
rupe ber kore ane.

Ei project-er ashol blob width (sigma = 2.67 px) diye synthetic blob-e jachai kora:

| heatmap noise | shadharon argmax | DARK | unnoti |
|---|---|---|---|
| shunno | 0.385 px | 0.000 px | nikhut |
| 0.02 | 0.475 px | 0.044 px | **10.8x** |
| 0.05 | 0.608 px | 0.154 px | **3.9x** |

**Shot sotorkota:** eগুলো adorsho Gaussian-e। Tomar model-er ashol blob erchaite ogocalo,
tai labh kom hobe. Kintu eta mapa bottleneck-kei aghat kore, ar cheshta korte **shunno khoroch**.

In [ ]:
def dark_refine(heatmap, peaks_xy, blur_ks=5):
    # Sub-pixel refinement of (x, y) peaks via DARK Taylor expansion.
    h = heatmap.astype(np.float64).copy()
    if blur_ks and blur_ks > 1:
        mx = h.max()
        if mx > 0:
            h = cv2.GaussianBlur(h, (blur_ks, blur_ks), 0)
            h = h * (mx / max(h.max(), 1e-12))
    h = np.log(np.maximum(h, 1e-10))
    H, W = h.shape
    out = []
    for (px, py) in peaks_xy:
        x, y = int(round(px)), int(round(py))
        if not (1 <= x < W - 1 and 1 <= y < H - 1):
            out.append([px, py]); continue
        dx  = 0.5 * (h[y, x+1] - h[y, x-1])
        dy  = 0.5 * (h[y+1, x] - h[y-1, x])
        dxx = h[y, x+1] - 2*h[y, x] + h[y, x-1]
        dyy = h[y+1, x] - 2*h[y, x] + h[y-1, x]
        dxy = 0.25 * (h[y+1, x+1] - h[y+1, x-1] - h[y-1, x+1] + h[y-1, x-1])
        det = dxx*dyy - dxy*dxy
        # 2-D maximum -> negative-definite Hessian: dxx < 0 AND det > 0
        if det > 1e-12 and dxx < 0:
            offset = -np.linalg.solve(np.array([[dxx, dxy], [dxy, dyy]]),
                                       np.array([dx, dy]))
            if np.abs(offset).max() < 1.0:
                out.append([x + offset[0], y + offset[1]]); continue
        out.append([float(x), float(y)])
    return np.array(out)


def evaluate_pia_net_dark(model, X_raw, meta, feat, batch=64, use_dark=True):
    preds = model.predict(X_raw, batch_size=batch, verbose=0)
    results = {}
    for i in range(len(X_raw)):
        Lp, SNR, QP = int(meta[i, 0]), int(meta[i, 1]), int(meta[i, 2])
        peaks, amps = get_blob_peaks(preds[i], detector)
        peaks = peaks[np.argsort(-amps)[:Lp]] if len(peaks) else peaks
        if use_dark and len(peaks):
            peaks = dark_refine(preds[i, :, :, 0], peaks)
        ang = peaks_to_angles(peaks, sigma=0.07, grid_size=preds.shape[1])
        gt_a, pr_a = prepare_for_metric(ang, feat[i])
        results.setdefault((Lp, SNR, QP), []).append((gt_a, pr_a))
    rmse, pd_ = {}, {}
    for cond, ex in results.items():
        good, bad = [], []
        for gt_a, pr_a in ex:
            if np.isnan(pr_a).any():
                bad.append(np.full(gt_a.size, 999.0)); continue
            g, b = filter_angles(get_ang_difference(gt_a, pr_a), 1.0)
            good.append(g); bad.append(b)
        good = np.concatenate(good) if good else np.array([])
        bad = np.concatenate(bad) if bad else np.array([])
        tot = len(good) + len(bad)
        rmse[cond] = np.sqrt(np.mean(good**2)) if len(good) else np.nan
        pd_[cond] = len(good)/tot if tot else np.nan
    return rmse, pd_


# ---- A/B comparison on the SAME trained weights ----
rmse_plain, pd_plain = evaluate_pia_net_dark(pia_net, X_test_raw, meta_test, feat_test, use_dark=False)
rmse_dark,  pd_dark  = evaluate_pia_net_dark(pia_net, X_test_raw, meta_test, feat_test, use_dark=True)

print('='*76)
print(f'{"SNR":>5} | {"RMSE plain":>10} {"RMSE DARK":>10} {"gain":>7} | {"Pd plain":>9} {"Pd DARK":>8}')
print('-'*76)
for c in sorted(rmse_plain, key=lambda k: k[1]):
    rp, rd = rmse_plain[c], rmse_dark[c]
    gain = rp/rd if (rd and not np.isnan(rd) and rd > 0) else float('nan')
    print(f'{c[1]:>5} | {rp:>10.4f} {rd:>10.4f} {gain:>6.2f}x | {pd_plain[c]:>9.4f} {pd_dark[c]:>8.4f}')
print('='*76)
mp = np.nanmean(list(rmse_plain.values())); md_ = np.nanmean(list(rmse_dark.values()))
pp = np.nanmean(list(pd_plain.values()));   pdk = np.nanmean(list(pd_dark.values()))
print(f'mean RMSE : {mp:.4f} -> {md_:.4f}   ({mp/md_:.2f}x better)')
print(f'mean Pd   : {pp:.4f} -> {pdk:.4f}   ({pdk-pp:+.4f})')
print()
print('UNet reference: mean RMSE = 0.3610, mean Pd = 0.7365')

# use the better decoding for the plots that follow
if md_ < mp:
    pia_rmse, pia_pd = rmse_dark, pd_dark
    print('\n-> DARK better, Part 13-er plot-e DARK result use kora hobe.')
else:
    print('\n-> DARK ei khetre labh dey ni, plain result-i rakha hocche.')

## Part 13 — Comparison plots vs the hardcoded baselines

In [ ]:
snrs = sorted(set(k[1] for k in UNET_REF_RMSE))
fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))

axs[0].plot(snrs, [UNET_REF_RMSE[(3,s,16)] for s in snrs], 'o-', label='UNet (reference)')
axs[0].plot(snrs, [RESNET_REF_RMSE[(3,s,16)] for s in snrs], 's-', label='ResNet (reference)')
if pia_rmse:
    ps = sorted(s for (l,s,q) in pia_rmse if l==3 and q==16)
    axs[0].plot(ps, [pia_rmse[(3,s,16)] for s in ps], '^-', color='crimson', label='PIA-Net (this run)')
axs[0].axhline(CHANCE_RMSE, ls='--', c='gray', label='chance floor (0.577)')
axs[0].set_xlabel('SNR (dB)'); axs[0].set_ylabel('RMSE (deg)'); axs[0].set_title('RMSE vs SNR')
axs[0].legend(); axs[0].grid(alpha=0.3)

axs[1].plot(snrs, [UNET_REF_PD[(3,s,16)] for s in snrs], 'o-', label='UNet (reference)')
axs[1].plot(snrs, [RESNET_REF_PD[(3,s,16)] for s in snrs], 's-', label='ResNet (reference)')
if pia_pd:
    ps = sorted(s for (l,s,q) in pia_pd if l==3 and q==16)
    axs[1].plot(ps, [pia_pd[(3,s,16)] for s in ps], '^-', color='crimson', label='PIA-Net (this run)')
axs[1].set_xlabel('SNR (dB)'); axs[1].set_ylabel('Pd'); axs[1].set_ylim(-0.02, 1.02)
axs[1].set_title('Detection probability vs SNR'); axs[1].legend(); axs[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
names = ['UNet', 'ResNet', 'PIA-Net']
params = [UNET_PARAMS, RESNET_PARAMS, PIA_PARAM_COUNT]
bars = ax.bar(names, params, color=['#4c72b0', '#dd8452', '#c44e52'])
ax.set_yscale('log'); ax.set_ylabel('Parameters (log)'); ax.set_title('Model size')
for b, p in zip(bars, params):
    ax.text(b.get_x()+b.get_width()/2, p, f'{p:,}', ha='center', va='bottom', fontsize=9)
plt.tight_layout(); plt.show()
print(f'PIA-Net is {UNET_PARAMS/PIA_PARAM_COUNT:.0f}x smaller than UNet, '
      f'{RESNET_PARAMS/PIA_PARAM_COUNT:.2f}x ResNet.')

## Part 14 — Final honest summary

Ei cell actual number diye summary banay। Jodi gap thake, seta lukay na — karon ei number-i
tumi supervisor-ke dekhabe.

In [ ]:
if pia_rmse:
    print('='*74)
    print(f'{"SNR":>5} | {"PIA RMSE":>9} {"PIA Pd":>8} | {"UNet Pd":>8} | {"gap":>7}')
    print('-'*74)
    gaps = []
    for s in sorted(s for (l,s,q) in pia_pd if l==3 and q==16):
        c = (3, s, 16)
        g = UNET_REF_PD[c] - pia_pd[c]
        gaps.append(g)
        print(f'{s:>5} | {pia_rmse[c]:>9.3f} {pia_pd[c]:>8.3f} | {UNET_REF_PD[c]:>8.3f} | {g:>+7.3f}')
    print('='*74)
    print(f'Mean Pd gap vs UNet: {np.nanmean(gaps):+.3f}')
    print(f'Model size: PIA-Net {PIA_PARAM_COUNT:,} vs UNet {UNET_PARAMS:,} '
          f'({UNET_PARAMS/PIA_PARAM_COUNT:.0f}x smaller)')
    print(f'Fresh training samples seen: {n_ep * STEPS_PER_EPOCH * BATCH_SIZE:,} '
          f'(UNet reference: ~5,000,000)')
    print()
    print('Defensible claim = "competitive accuracy at a fraction of the model size",')
    print('NOT "beats UNet on accuracy" -- unless the numbers above actually show that.')
else:
    print('no evaluation available.')